# MobiHet V5 G
Frozen input contracts; DAGSNet-only. No raw/perturbed/latent/per-record gradients exported; aggregate stats >=5 or noise-init synthetic exception. Teacher H first, private G/J; r20 internal holdouts, ten metrics from mean client confusion. W&B authenticated online before train. Books capped at 2x measured comparable baseline. Not R1/R2/crypto/DP proof.

In [1]:

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
SEED=43
CODE=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
count=torch.cuda.device_count()
assert count==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(count))
assert torch.__version__=='2.10.0+cu128',torch.__version__
print('torch',torch.__version__,'gpus',[torch.cuda.get_device_name(i) for i in range(count)],flush=True)
hits=[p.parent for p in Path('/kaggle/input').rglob('v2_run.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0]; WORK=Path('/kaggle/working/work'); OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source'); SOURCE.mkdir(exist_ok=True)
for name in ('mh','knowledge','papers/pilot/fixture'): (WORK/name).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for f in SRC.iterdir():
    if f.name in CODE: dst=WORK/'mh'/f.name
    elif f.name=='meta.json': dst=WORK/'knowledge'/f.name
    elif f.name.startswith('pilot-v5-') and f.suffix=='.md': continue
    else: dst=WORK/'papers/pilot/fixture'/f.name
    shutil.copyfile(f,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
print('verified source and fixture',len(manifest),len(fixture['files']),flush=True)
sys.path.insert(0,str(WORK))
import numpy as np
from mh.dagsnet import build_model,CFG
from mh.v2_math import Layout,CountSketch,SeedBasis,unpack
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device); torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device).eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy')[:32].astype('float32')).to(device)
    y=torch.from_numpy(np.load(SRC/'train_0_y.npy')[:32].astype('int64')).to(device)
    layout=Layout(model); assert layout.size==395024
    with torch.autocast('cuda',dtype=torch.float16): z=model(x)[0].float()
    loss=torch.nn.functional.cross_entropy(z,y); assert torch.isfinite(loss); loss.backward()
    gradient=torch.cat([p.grad.float().reshape(-1) for p in model.parameters()])
    assert torch.isfinite(gradient).all()
    sketch=CountSketch(layout.size,3,256,1701,device)
    packet,delta=sketch.patch(sketch.encode(gradient),128,0); assert torch.isfinite(delta).all()
    assert unpack(packet,device,0)['version']==1
    basis=SeedBasis(model,8,1901,device); coeff=torch.zeros(8,device=device,requires_grad=True)
    torch.nn.functional.cross_entropy(basis.forward(model,coeff,x)[0].float(),y).backward()
    assert torch.isfinite(coeff.grad).all()
    cm=torch.bincount(y*16+z.argmax(1),minlength=256); assert int(cm.sum())==len(y)
    print('T4 real-model deterministic preflight PASS',gpu,flush=True)
    del model,x,y,z,gradient,sketch,basis,coeff,delta; torch.cuda.empty_cache()
torch.cuda.set_device(0)

import copy
from types import SimpleNamespace
from mh.v5_books import Method as Books
torch.set_num_threads(1)
_book_times={1:{'mean':[],'grad':[]},4:{'mean':[],'grad':[]}}
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device); torch.manual_seed(SEED)
    source_y=np.load(SRC/'train_0_y.npy'); source_x=np.load(SRC/'train_0_X.npy',mmap_mode='r')
    indices=np.concatenate([np.flatnonzero(source_y==c)[:32] for c in range(16)])
    x=torch.from_numpy(np.array(source_x[indices])).to(device).float()
    y=torch.from_numpy(np.array(source_y[indices])).to(device).long()
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device)
    mu=torch.stack([x[y==c].mean(0) for c in range(16)])
    sd=torch.stack([x[y==c].var(0,correction=0).clamp_min(1e-4).sqrt() for c in range(16)])
    for ipc in (1,4):
      for objective in ('mean','grad'):
        run=SimpleNamespace(init=model,priv=[copy.deepcopy(model)],dev=device,K=1,bytes=[],X=[x],Y=[y],n=[len(y)],
             a=SimpleNamespace(arm=f'g{ipc}_{objective}_trained',smoke=False,seed=SEED))
        run.gen=lambda *keys: torch.Generator(device=device).manual_seed(int(np.random.SeedSequence([SEED,*keys]).generate_state(1)[0]))
        method=Books(run,initialize=False); method.mu=mu.half().float(); method.sd=sd.half().float()
        torch.cuda.synchronize(); started=time.perf_counter(); book,diagnostic=method.condense(0,1,steps=80); torch.cuda.synchronize()
        seconds=time.perf_counter()-started
        assert diagnostic['slots']==diagnostic['applied']==80 and torch.isfinite(book).all()
        _book_times[ipc][objective].append(seconds)
        print('T4 V5 full80 book preflight PASS',gpu,ipc,objective,round(seconds,3),flush=True)
        del method,run,book,diagnostic
    del model,x,y,mu,sd,source_x,source_y,indices
    torch.cuda.empty_cache()
(SOURCE/'book_preflight.json').write_text(json.dumps({'seconds':_book_times,'seed':SEED,
    'evidence':'one producer <=32rows/class, initial DAGSNet; full80 steps both objectives/IPC/devices, no F1/mixing/current-trained guarantee'}))
torch.cuda.set_device(0)


torch 2.10.0+cu128 gpus ['Tesla T4', 'Tesla T4']
verified source and fixture 28 66
T4 real-model deterministic preflight PASS 0
T4 real-model deterministic preflight PASS 1
T4 V5 full80 book preflight PASS 0 1 mean 1.41
T4 V5 full80 book preflight PASS 0 1 grad 4.241
T4 V5 full80 book preflight PASS 0 4 mean 1.124
T4 V5 full80 book preflight PASS 0 4 grad 4.303
T4 V5 full80 book preflight PASS 1 1 mean 1.103
T4 V5 full80 book preflight PASS 1 1 grad 4.232
T4 V5 full80 book preflight PASS 1 4 mean 1.073
T4 V5 full80 book preflight PASS 1 4 grad 4.185


In [3]:
SEED=43
METHODS=['g1_diag', 'g1_noise', 'g1_mean_frozen', 'g1_mean_trained', 'g1_grad_frozen', 'g1_grad_trained', 'g1_own']

OUT.mkdir(exist_ok=True); baseline=list(METHODS[:2]); queue=list(METHODS[2:])
assert baseline[0].endswith('_diag') and baseline[1].endswith('_noise')
slots=[0,0,1,1]; active={}; done={}; seen={}; costs={}; budget=None; baseline_done=False
T0=time.monotonic()
def not_run(arm,reason):
    d=OUT/f'{arm}_s{SEED}'; d.mkdir(exist_ok=True)
    identity=json.loads((OUT/f'{METHODS[0]}_s{SEED}'/'config.json').read_text())
    identity.pop('fingerprint'); identity['config']['arm']=arm
    identity['fingerprint']=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    (d/'config.json').write_text(json.dumps(identity)); (d/'log.txt').write_text(reason+'\n')
    done[arm]={'rc':-998,'complete':False,'not_started':True,'reason':reason}
while baseline or queue or active:
    if not baseline_done and all(arm in done for arm in METHODS[:2]):
        baseline_done=True
        if not all(done[arm]['complete'] and done[arm]['rc']==0 for arm in METHODS[:2]):
            for arm in queue: not_run(arm,'COST BASELINE MISSING: no valid comparable diagonal timing')
            queue=[]
        else:
            base=json.loads((OUT/f'{METHODS[0]}_s{SEED}'/'result.json').read_text())
            baseline_seconds=base['minutes']*60; budget=2*baseline_seconds
            ipc=1 if METHODS[0].startswith('g1_') else 4
            for arm in list(queue):
                objective='grad' if 'grad' in arm or arm.endswith('_own') else 'mean'
                projected=baseline_seconds+2*1.15*max(_book_times[ipc][objective])*20*4+60
                if projected>budget:
                    not_run(arm,'COST PREFLIGHT REJECT: projected total exceeds 2x measured baseline'); queue.remove(arm)
            (SOURCE/'cost_reference.json').write_text(json.dumps(dict(baseline_seconds=baseline_seconds,
                max_seconds=budget,baseline_arm=METHODS[0],concurrency='baseline diag+noise onGPU0; cases two workers/device',
                prediction_rule='baseline + 2*1.15*single_producer_book_seconds*20*4 +60')))
    for slot,gpu in enumerate(slots):
        if slot in active: continue
        if not baseline_done:
            if slot>1 or not baseline: continue
            arm=baseline.pop(0); limit=None
        else:
            if not queue: continue
            arm=queue.pop(0); limit=budget
        d=OUT/f'{arm}_s{SEED}'; d.mkdir(exist_ok=True); log=(d/'log.txt').open('w')
        env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
        command=[sys.executable,'-m','mh.v5_run','--arm',arm,'--seed',str(SEED),'--out',str(OUT),
                 '--wandb_online','--max_hours',str(limit/3600 if limit is not None else 4)]
        active[slot]=dict(arm=arm,proc=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT),
                          log=log,start=time.monotonic(),limit=limit,terminated=False)
    for slot,item in list(active.items()):
        arm=item['arm']; proc=item['proc']; d=OUT/f'{arm}_s{SEED}'
        lines=(d/'log.txt').read_text().splitlines()
        for line in lines[seen.get(arm,0):]:
            if line.startswith(('setup','r','done','WANDB','Traceback','budget')) or 'Error' in line: print(arm,'|',line,flush=True)
        seen[arm]=len(lines)
        elapsed=time.monotonic()-item['start']
        if item['limit'] is not None and elapsed>=item['limit']-30 and proc.poll() is None:
            if not item['terminated']:
                print('COST CAP terminate',arm,round(elapsed),flush=True); proc.terminate(); item['terminated']=True
            elif elapsed>=item['limit']-10: proc.kill()
        if proc.poll() is not None:
            result=json.loads((d/'result.json').read_text()) if (d/'result.json').exists() else {}
            active_seconds=result.get('minutes',elapsed/60)*60
            cost_pass=item['limit'] is None or (active_seconds<=item['limit'] and not item['terminated'])
            done[arm]={'rc':proc.returncode,'complete':result.get('complete',False),'cost_stopped':item['terminated'],'cost_pass':cost_pass}
            costs[arm]=dict(worker_wall_seconds=elapsed,active_seconds=active_seconds,
                            max_seconds=item['limit'],cost_stopped=item['terminated'],cost_pass=cost_pass)
            item['log'].close(); del active[slot]
    if baseline or queue or active: time.sleep(5)
(OUT/f'summary_s{SEED}.json').write_text(json.dumps(done,indent=2))
(SOURCE/'cost_observed.json').write_text(json.dumps(costs,indent=2))
print(json.dumps(done),flush=True)
shutil.rmtree(WORK)
needle=os.environ['WANDB_API_KEY'].encode()
for root in (OUT,SOURCE):
  for path in root.rglob('*'):
    if not path.is_file(): continue
    carry=b''
    with path.open('rb') as handle:
      while True:
        part=handle.read(1024*1024)
        if not part: break
        assert needle not in carry+part,('credential in output',path.name)
        carry=(carry+part)[-len(needle):]
assert set(done)==set(METHODS) and all(v['rc']==0 and v['complete'] and v.get('cost_pass',True) for v in done.values()),done


g1_diag | WANDB https://wandb.ai/21522798-uit/mobihet-v5/runs/g1_diag_s43-2245a273
g1_diag | setup g1_diag role private rows 1000369 sec 1.5
g1_noise | WANDB https://wandb.ai/21522798-uit/mobihet-v5/runs/g1_noise_s43-9e21da38
g1_noise | setup g1_noise role private rows 1000369 sec 1.4
g1_diag | r1 g1_diag s43 private group gp None loss 2.3531 91s
g1_noise | r1 g1_noise s43 private group gp None loss 0.9981 91s
g1_diag | r2 g1_diag s43 private group gp None loss 1.4593 90s
g1_noise | r2 g1_noise s43 private group gp None loss 0.4440 90s
g1_diag | r3 g1_diag s43 private group gp None loss 1.2510 90s
g1_noise | r3 g1_noise s43 private group gp None loss 0.3362 90s
g1_diag | r4 g1_diag s43 private group gp None loss 1.1302 90s
g1_noise | r4 g1_noise s43 private group gp None loss 0.2716 90s
g1_diag | r5 g1_diag s43 private group gp 0.5530259064537687 loss 1.0444 121s
g1_noise | r5 g1_noise s43 private group gp 0.4842400836306681 loss 0.2233 120s
g1_diag | r6 g1_diag s43 private group gp No